In [1]:
from arcgis.gis import GIS
from datetime import datetime
from datetime import timezone
now = datetime.now(timezone.utc)

In [2]:
gis_premium = GIS("https://www.arcgis.com", "hubpy_test", "hubPython01")
myhub = gis_premium.hub
gis_portal = GIS("https://rpubs22001.ags.esri.com/portal/home/", "creator1", "portalaccount1")

### Add Initiative

In [3]:
#Add initiative
title = "Test initiative %s" %int(now.timestamp() * 1000)
new_initiative = myhub.initiatives.add(title=title)
initiative_id = new_initiative.itemid
assert new_initiative.title==title, new_initiative.title

### Add page to this site

In [4]:
title = "Test page %s" %int(now.timestamp() * 1000)
#Fetch site object to add page to
new_site = myhub.sites.get(new_initiative.site_id)
#Add new page
new_page = new_site.pages.add(title=title)
assert new_page.title==title, new_page.title

### Search for added page

In [5]:
#Searching for page
searched = myhub.pages.search(title=title, owner=gis_premium.users.me.username)
assert searched[0].itemid==new_page.itemid, searched.item

### Get page

In [6]:
#Fetching page
fetched = myhub.pages.get(new_page.itemid)
assert fetched==searched[0], fetched

### Link Page

In [7]:
#Fetch page
page1 = myhub.pages.get('1c75da020b2f4943a3f657ef59fc1bc8')
#Link page
new_site.pages.link(page1)
pages = new_site.pages.search()
assert len(pages)==2, pages

### Clone page

In [8]:
page_cloned = new_site.pages.clone(new_page)
pages = new_site.pages.search()
assert len(pages)==3, pages

### Unlink page

In [9]:
new_site.pages.unlink(page1)
pages = new_site.pages.search()
assert len(pages)==2, pages

### Delete page

In [10]:
page_cloned.delete()
new_site = myhub.sites.get(new_site.itemid)
pages = new_site.pages.search()
assert len(pages)==1, pages

# Testing for Enterprise sites

In [11]:
#Adding new site in Portal organization
title = "New site %s" %int(now.timestamp() * 1000)
portal_site = gis_portal.sites.add(title=title)
site_id = portal_site.itemid
assert portal_site.title==title, portal_site.title

### Adding page to new site

In [12]:
#Adding page to this site
title = "Test page %s" %int(now.timestamp() * 1000)
#Add new page
portal_page = portal_site.pages.add(title=title)
assert portal_page.title==title, portal_page.title

### Updating page

In [13]:
assert portal_page.tags==[]
portal_page.update(page_properties={'tags': 'Enterprise Sites'})
assert 'Enterprise Sites' in portal_page.tags, portal_page.tags

### Clone page from Hub to Enterprise

In [14]:
page2 = portal_site.pages.clone(new_page)
pages = portal_site.pages.search()
assert len(pages)==2, len(pages)

### Delete initiative and site

In [15]:
def verify_deleted_group(gis, g_id):
    '''
    Verify group is deleted
    '''
    if gis.groups.get(g_id) is None:
        return 'Works as expected'
    

def verify_deleted_item(gis, i_id):
    '''
    Verify item is deleted
    '''
    if gis.content.get(i_id) is None:
        return 'Works as expected'

In [16]:
#Delete premium initiative (deletes page too since it isn't linked to any other site)
site_id = new_initiative.site_id
initiative_id = new_initiative.itemid
page_id = new_page.itemid
collab_group_id = new_initiative.collab_group_id
content_group_id = new_initiative.content_group_id
followers_group_id = new_initiative.followers_group_id
new_initiative.delete()
assert verify_deleted_item(gis_premium, initiative_id)=='Works as expected', 'initiative exists'
assert verify_deleted_item(gis_premium, site_id)=='Works as expected', 'site exists'
assert verify_deleted_item(gis_premium, page_id)=='Works as expected', 'page exists'
assert verify_deleted_group(gis_premium, collab_group_id)=='Works as expected', 'collab group exists'
assert verify_deleted_group(gis_premium, content_group_id)=='Works as expected', 'content group exists'
assert verify_deleted_group(gis_premium, followers_group_id)=='Works as expected', 'followers group exists'

In [17]:
#Delete enterprise site (deletes both pages since they aren't linked to other sites)
site_id = portal_site.itemid
page_id = portal_page.itemid
cloned_page_id = page2.itemid
content_group_id = new_site.content_group_id
portal_site.delete()
assert verify_deleted_item(gis_portal, site_id)=='Works as expected', 'site exists'
assert verify_deleted_item(gis_portal, page_id)=='Works as expected', 'page exists'
assert verify_deleted_item(gis_portal, cloned_page_id)=='Works as expected', 'cloned page exists'
assert verify_deleted_group(gis_portal, content_group_id)=='Works as expected', 'content group exists'